### Text To Sql Agent

In [19]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine
import os 
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [21]:
DB_USER=os.getenv("DB_USER")
DB_NAME=os.getenv("DB_NAME")
DB_PASSWORD=os.getenv("DB_PASSWORD")
DB_PORT=os.getenv("DB_PORT")
DB_HOST=os.getenv("DB_HOST")

In [22]:
from sqlalchemy.engine import URL

connection_url=URL.create(
    "mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

In [23]:
engine=create_engine(
    connection_url
)

In [24]:
db=SQLDatabase(
    engine,
    sample_rows_in_table_info=0,
    ignore_tables=["_demodb_import"]
)

In [25]:
def get_schema():
    return db.get_table_info()

In [26]:
schema=get_schema()

In [27]:
print(schema)


CREATE TABLE album (
	`AlbumId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Title` VARCHAR(160) NOT NULL, 
	`ArtistId` BIGINT NOT NULL, 
	PRIMARY KEY (`AlbumId`), 
	CONSTRAINT album_ibfk_1 FOREIGN KEY(`ArtistId`) REFERENCES artist (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE artist (
	`ArtistId` BIGINT NOT NULL AUTO_INCREMENT, 
	`Name` VARCHAR(120), 
	PRIMARY KEY (`ArtistId`)
)ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE utf8mb4_0900_ai_ci


CREATE TABLE customer (
	`CustomerId` BIGINT NOT NULL AUTO_INCREMENT, 
	`FirstName` VARCHAR(40) NOT NULL, 
	`LastName` VARCHAR(20) NOT NULL, 
	`Company` VARCHAR(80), 
	`Address` VARCHAR(70), 
	`City` VARCHAR(40), 
	`State` VARCHAR(40), 
	`Country` VARCHAR(40), 
	`PostalCode` VARCHAR(10), 
	`Phone` VARCHAR(24), 
	`Fax` VARCHAR(24), 
	`Email` VARCHAR(60) NOT NULL, 
	`SupportRepId` BIGINT, 
	PRIMARY KEY (`CustomerId`), 
	CONSTRAINT customer_ibfk_1 FOREIGN KEY(`SupportRepId`) REFERENCES employee (`EmployeeId`)


In [28]:
def run_query(query):
    print(f"Query being executed:{query}")
    return db.run(query)

In [29]:
run_query("""
SELECT Artist.Name
FROM Artist
LIMIT 5;
""")

Query being executed:
SELECT Artist.Name
FROM Artist
LIMIT 5;



"[('AC/DC',), ('Accept',), ('Aerosmith',), ('Alanis Morissette',), ('Alice In Chains',)]"

### LLM Initilization

In [30]:
from langchain_groq import ChatGroq

GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [31]:
llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=GROQ_API_KEY
)

### SQL Generator

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def write_sql_query(llm):

    template="""
        Based on the table schema below , write a SQL Query
        that would answer the user's question.

        Schema:{schema}

        Question:{question}

        SQL Query:
    """

    prompt=ChatPromptTemplate.from_messages([
        ("system","""
            You are an expert SQL Query generator.
            Generate only valid MySQL SQL Queries.
            Do not include explainations or markdowns.
        """),
        ("human",template)
    ])

    return prompt|llm|StrOutputParser()

In [33]:
# Test generation

sql_chain=write_sql_query(llm)

response=sql_chain.invoke({
    "schema":get_schema(),
    "question":"Show all tracks by AC/DC"
})

print(response)

SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';


In [34]:
result=run_query(response)

print(result)

Query being executed:SELECT t.TrackId, t.Name AS TrackName, a.Title AS AlbumTitle
FROM track t
JOIN album a ON t.AlbumId = a.AlbumId
JOIN artist ar ON a.ArtistId = ar.ArtistId
WHERE ar.Name = 'AC/DC';
[(1, 'For Those About To Rock (We Salute You)', 'For Those About To Rock We Salute You'), (6, 'Put The Finger On You', 'For Those About To Rock We Salute You'), (7, "Let's Get It Up", 'For Those About To Rock We Salute You'), (8, 'Inject The Venom', 'For Those About To Rock We Salute You'), (9, 'Snowballed', 'For Those About To Rock We Salute You'), (10, 'Evil Walks', 'For Those About To Rock We Salute You'), (11, 'C.O.D.', 'For Those About To Rock We Salute You'), (12, 'Breaking The Rules', 'For Those About To Rock We Salute You'), (13, 'Night Of The Long Knives', 'For Those About To Rock We Salute You'), (14, 'Spellbound', 'For Those About To Rock We Salute You'), (15, 'Go Down', 'Let There Be Rock'), (16, 'Dog Eat Dog', 'Let There Be Rock'), (17, 'Let There Be Rock', 'Let There Be Rock

### SQL Validator

In [35]:
import sqlglot
from sqlglot import exp

In [39]:
def validate_sql(query):

    try:
        parsed=sqlglot.parse(query,read="mysql")

        if len(parsed)!=1:
            return {
                "valid":False,
                "operation":None,
                "message":"Only one SQL Statement is allowed"
            }

        statement=parsed[0]

        if isinstance(statement,exp.Select):
            operation="SELECT"

        elif isinstance(statement,exp.Insert):
            operation="INSERT"

        elif isinstance(statement,exp.Update):
            operation="UPDATE"

        elif isinstance(statement,exp.Delete):
            operation="DELETE"

        elif isinstance(statement,exp.Alter):
            operation="ALTER"

        elif isinstance(statement,exp.Drop):
            operation="ALTER"

        elif isinstance(statement,exp.Create):
            operation="CREATE"

        else:
            return {
                "Valid":False,
                "operation":None,
                "message":"Unsupported SQL operation"
            }

        return {
            "valid":True,
            "operation":operation,
            "message":f"{operation} query validated"
        }
    except Exception as e:
        return {
            "valid":False,
            "operation":None,
            "message":str(e)
        }


In [41]:
print(validate_sql("SELECT * FROM Track"))
print(validate_sql("DROP TABLE Track"))
print(validate_sql("DELETE FROM Track"))
print(validate_sql("UPDATE Track SET Name = 'Test'"))

{'valid': True, 'operation': 'SELECT', 'message': 'SELECT query validated'}
{'valid': True, 'operation': 'ALTER', 'message': 'ALTER query validated'}
{'valid': True, 'operation': 'DELETE', 'message': 'DELETE query validated'}
{'valid': True, 'operation': 'UPDATE', 'message': 'UPDATE query validated'}


### SQL Approval

In [45]:
def check_approval(operation):

    if operation=="SELECT":
        return False

    return True

In [47]:
operations=["DELETE","SELECT","INSERT","ALTER","UPDATE"]

for operation in operations:

    approval=check_approval(operation)

    if approval:
        print(operation,"-> Human approval required")

    else:
        print(operation,"->No approval required")

DELETE -> Human approval required
SELECT ->No approval required
INSERT -> Human approval required
ALTER -> Human approval required
UPDATE -> Human approval required


In [50]:
def prepare_sql(query):

    validation=validate_sql(query)

    if not validation['valid']:
        return {
            "status":"rejected",
            "query":query,
            "message":validation['message']
        }

    operation=validation['operation']

    approval_required=check_approval(operation)

    if approval_required:
        return {
            "status":"approval_required",
            "query":query,
            "operation":operation,
            "message":"Human Approval Required"
        }

    return {
        "status":"ready",
        "query":query,
        "operation":operation,
        "message":"Query is ready for execution"
    }


In [51]:
print(prepare_sql("SELECT * FROM Track LIMIT 5"))

print(prepare_sql(
    "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1"
))

print(prepare_sql("DROP TABLE Track"))

{'status': 'ready', 'query': 'SELECT * FROM Track LIMIT 5', 'operation': 'SELECT', 'message': 'Query is ready for execution'}
{'status': 'approval_required', 'query': "UPDATE Artist SET Name = 'Updated Artist' WHERE ArtistId = 1", 'operation': 'UPDATE', 'message': 'Human Approval Required'}
{'status': 'approval_required', 'query': 'DROP TABLE Track', 'operation': 'ALTER', 'message': 'Human Approval Required'}


### Agent

In [55]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Optional

In [56]:
class AgentState(TypedDict):

    question: str
    schema: str
    sql_query: str
    operation: Optional[str]
    status: str
    approval_required: bool
    result: Optional[str]

In [57]:
# SQL Generator Node

def generate_sql_node(state:AgentState):

    response=sql_chain.invoke({
        "schema":state["schema"],
        "question":state["question"]
    })

    return {
        "sql_query":response
    }

In [59]:
test_state = {
    "question":"Show me all tracks by AC/DC",
    "schema":get_schema(),
    "sql_query":"",
    "operation":None,
    "status":"started",
    "approval_required":False,
    "result":None
}

output=generate_sql_node(test_state)

print(output)

{'sql_query': "SELECT t.`Name` AS TrackName, a.`Title` AS AlbumTitle, ar.`Name` AS ArtistName\nFROM `track` t\nJOIN `album` a ON t.`AlbumId` = a.`AlbumId`\nJOIN `artist` ar ON a.`ArtistId` = ar.`ArtistId`\nWHERE ar.`Name` = 'AC/DC'\nORDER BY t.`Name`;"}
